In [27]:
from datasets import load_dataset
from datasets import get_dataset_config_names

import pandas as pd

from pathlib import Path

In [2]:
ru_corpus = load_dataset(
    "kaengreg/sberquad-retrieval",
    "corpus",
    split="train"
)
ru_corpus

Dataset({
    features: ['_id', 'title', 'text', 'processed_text', 'processed_title'],
    num_rows: 17474
})

In [4]:
df_ru_corpus = ru_corpus.to_pandas()

print(df_ru_corpus.shape)

df_ru_corpus.head()

(17474, 5)


,_id,title,text,processed_text,processed_title
0,0,,В протерозойских отложениях органические остат...,протерозойский отложение органический остаток ...,
1,1,,Кишечник млекопитающего подразделяется на тонк...,кишечник млекопитающее подразделяться тонкий т...,
2,2,,Город Байконур и космодром Байконур вместе обр...,город байконур космодром байконур вместе образ...,
3,3,,Вскоре после прибытия Колумба из Вест-Индии во...,вскоре прибытие колумб вест индия возникнуть н...,
4,4,,Около Порт-Артура ночью на 27 января 1904 года...,около порт артур ночью 27 январь 1904 год нача...,


In [5]:
df_ru_corpus.info()

<class 'pandas.DataFrame'>
RangeIndex: 17474 entries, 0 to 17473
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   _id              17474 non-null  int64
 1   title            17474 non-null  str  
 2   text             17474 non-null  str  
 3   processed_text   17474 non-null  str  
 4   processed_title  17474 non-null  str  
dtypes: int64(1), str(4)
memory usage: 42.8 MB


In [6]:
df_ru_corpus.isnull().sum()

_id                0
title              0
text               0
processed_text     0
processed_title    0
dtype: int64

In [7]:
df_ru_corpus["text"].duplicated().sum()

np.int64(3985)

In [8]:
duplicate_texts = df_ru_corpus[
    df_ru_corpus["text"].duplicated(keep=False)
]

duplicate_texts[
    ["title", "text"]
].head(10)

,title,text
2,,Город Байконур и космодром Байконур вместе обр...
5,,"Например, вашингтонский Hera Hub — это коворки..."
10,,Корпоративный сайт — содержит полную информаци...
11,,Звук в старинной трубе извлекался свободно и с...
12,,Скорость света в вакууме — абсолютная величина...
16,,Дальнейшее развитие качества киноизображения с...
21,,"Находясь в зените своей славы, Моцарт получает..."
22,,"По характеру пищи, используемой в процессе жиз..."
23,,"В большинстве стран, в отличие от стран СНГ, н..."
24,,Ведущие области промышленности: машиностроител...


In [9]:
(df_ru_corpus["title"] == "").sum()

np.int64(17474)

In [10]:
(df_ru_corpus["processed_title"] == "").sum()

np.int64(17474)

In [11]:
df_ru_corpus["text_length"] = df_ru_corpus["text"].str.len()

df_ru_corpus["text_length"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    17474.000000
mean       751.979799
std        253.256310
min        279.000000
50%        681.000000
75%        856.000000
90%       1072.700000
95%       1204.000000
99%       1596.160000
max       7231.000000
Name: text_length, dtype: float64

In [12]:
df_ru_corpus.nlargest(
    5,
    "text_length"
)[["_id", "text_length", "text"]]

,_id,text_length,text
8178,8178,7231,Во Франции пластика продолжала держаться парад...
5700,5700,4027,"В Германии, после Торвальдсена, в числе скульп..."
4311,4311,3489,"Крупные предприятия города: ПО Гомсельмаш , РУ..."
11537,11537,3489,"Крупные предприятия города: ПО Гомсельмаш , РУ..."
2213,2213,3412,За участие в кампании 1809 г. против Австрии Б...


In [13]:
df_ru_corpus["text"].nunique()

13489

In [14]:
(
    df_ru_corpus["text"]
    ==
    df_ru_corpus["processed_text"]
).mean()

np.float64(0.0)

In [15]:
df_ru_corpus[
    ["text", "processed_text"]
].sample(5, random_state=42)

,text,processed_text
9585,Но в древние времена люди видели во взаимном р...,древний время человек видеть взаимный располож...
14777,"Родившись в качестве жаргонизма, синонима назв...",родиться качество жаргонизм синоним название м...
247,Во время анафазы В расходятся сами полюса деле...,время анафаза расходиться полюс деление клетка...
14068,В любовной лирике Тютчев создаёт ряд стихотвор...,любовный лирика тютчев создавать ряд стихотвор...
6630,Ариперт I (653—661) занял престол при поддержк...,ариперт i 653 661 занять престол поддержка кат...


In [16]:
df_ru_corpus["processed_text_length"] = (
    df_ru_corpus["processed_text"]
    .str.len()
)

df_ru_corpus["processed_text_length"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    17474.000000
mean       629.236237
std        212.054904
min        220.000000
50%        572.000000
75%        716.000000
90%        896.700000
95%       1002.000000
99%       1342.270000
max       5948.000000
Name: processed_text_length, dtype: float64

In [17]:
ru_queries = load_dataset(
    "kaengreg/sberquad-retrieval",
    "queries",
    split="train"
).to_pandas()

print(ru_queries.shape)

ru_queries.head()

(74300, 5)


,_id,text,processed_text,title,processed_title
0,100,чем представлены органические остатки?,представить органический остаток,,
1,101,что найдено в кремнистых сланцах железорудной ...,найти кремнистый сланец железорудный формация ...,,
2,102,что встречается в протерозойских отложениях?,встречаться протерозойский отложение,,
3,103,что относится к числу древнейших растительных ...,относиться число древний растительный остаток,,
4,104,как образовалось графито-углистое вещество?,образоваться графито углистый вещество,,


In [18]:
(ru_queries["title"] == "").sum()

np.int64(74300)

In [19]:
ru_queries["text"].duplicated().sum()

np.int64(16)

In [20]:
ru_queries.columns.tolist()

['_id', 'text', 'processed_text', 'title', 'processed_title']

In [22]:
get_dataset_config_names("kaengreg/sberquad-retrieval")

['corpus', 'queries']

In [23]:
ru_queries.info()

<class 'pandas.DataFrame'>
RangeIndex: 74300 entries, 0 to 74299
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   _id              74300 non-null  int64
 1   text             74300 non-null  str  
 2   processed_text   74300 non-null  str  
 3   title            74300 non-null  str  
 4   processed_title  74300 non-null  str  
dtypes: int64(1), str(4)
memory usage: 18.2 MB


In [24]:
ru_queries.sample(5, random_state=42)

,_id,text,processed_text,title,processed_title
8400,8500,"Что привело к появлению мнения о том, что терм...",привести появление мнение термин оппортунизм я...,,
4293,4393,"У каких бабочек, к примеру, яркая окраска крыл...",бабочка пример яркий окраска крыло контролиров...,,
38171,38271,Под чьим воздействием формируются простые угле...,чей воздействие формироваться простой углеводород,,
26063,26163,Какой год был самым важным для формирования Пр...,год самый важный формирование пруст писателя7,,
47980,48080,"Как зовут итальянского футболиста Ривера, заби...",звать итальянский футболист ривер забить побед...,,


In [25]:
print("Corpus:")
print(df_ru_corpus["_id"].min(), df_ru_corpus["_id"].max())

print("\nQueries:")
print(ru_queries["_id"].min(), ru_queries["_id"].max())

Corpus:
0 17473

Queries:
100 74399


In [32]:
print("Corpus IDs:")
print(df_ru_corpus["_id"].head(20).tolist())

print("\nQuery IDs:")
print(ru_queries["_id"].head(20).tolist())

Corpus IDs:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Query IDs:
[100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119]


In [33]:
dataset_dict = load_dataset(
    "kaengreg/sberquad-retrieval",
    "queries"
)

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['_id', 'text', 'processed_text', 'title', 'processed_title'],
        num_rows: 74300
    })
})

In [34]:
dataset_dict["train"].features

{'_id': Value('int64'),
 'text': Value('string'),
 'processed_text': Value('string'),
 'title': Value('string'),
 'processed_title': Value('string')}

In [35]:
corpus_dict = load_dataset(
    "kaengreg/sberquad-retrieval",
    "corpus"
)

corpus_dict["train"].features

{'_id': Value('int64'),
 'title': Value('string'),
 'text': Value('string'),
 'processed_text': Value('string'),
 'processed_title': Value('string')}